In [1]:
from cmdstanpy import CmdStanModel
import numpy as np
import pandas as pd
from pathlib import Path
import scipy.stats as st
import os

from joblib import Parallel, delayed
from tqdm.auto import tqdm

from deconveil.nb_regression_fit import *

#### Compile Stan model

In [5]:
stan_path = Path("/Users/katsiarynadavydzenka/Documents/PhD_AI/DeConveil/stan/NB_regression_puradj.stan")
model = CmdStanModel(stan_file=str(stan_path))

#### Fit one gene

In [29]:
DATA_PATH = "/Users/katsiarynadavydzenka/Documents/PhD_AI/DeConveil/datasets/tcga_luad/"
df_long = pd.read_csv(os.path.join(DATA_PATH, "stan_joint_long.csv"))
df_long.head()

,gene,sample_id,expr,copies,purity,stroma,sf,eup_dev_cancer,eup_equiv_cancer
0,BRAF,TCGA-50-5932-01A,1121,1,0.83,0.17,1.161082,-0.5,0.5
1,BRAF,TCGA-49-6742-01A,474,2,0.74,0.26,0.765953,0.0,1.0
2,BRAF,TCGA-44-6147-01A,2347,3,0.79,0.21,1.196192,0.5,1.5
3,BRAF,TCGA-55-6979-01A,902,2,0.34,0.66,0.826870,0.0,1.0
4,BRAF,TCGA-50-5931-01A,929,3,0.83,0.17,0.967765,0.5,1.5


In [31]:
# Fit 
gene_df = df_long[df_long["gene"] == "BRAF"]
res_singleGene = fit_one_gene(gene_df, 
                   model, 
                   et=0.15, 
                   min_aneup=5,
                   cna="all")

12:11:50 - cmdstanpy - INFO - CmdStan start processing
12:11:50 - cmdstanpy - INFO - Chain [1] start processing
12:11:50 - cmdstanpy - INFO - Chain [2] start processing
12:11:50 - cmdstanpy - INFO - Chain [3] start processing
12:11:50 - cmdstanpy - INFO - Chain [4] start processing
12:22:38 - cmdstanpy - INFO - Chain [1] done processing
12:22:43 - cmdstanpy - INFO - Chain [2] done processing
12:22:44 - cmdstanpy - INFO - Chain [3] done processing
12:22:45 - cmdstanpy - INFO - Chain [4] done processing
12:22:45 - cmdstanpy - WARNING - Non-fatal error during sampling:
Exception: neg_binomial_2_lpmf: Location parameter[1] is inf, but must be positive finite! (in 'NB_regression_puradj.stan', line 76, column 2 to column 30)
Consider re-running with show_console=True if the above output is unclear!
12:22:45 - cmdstanpy - WARNING - Some chains may have failed to converge.
	Chain 1 had 998 iterations at max treedepth (99.8%)
	Chain 2 had 999 iterations at max treedepth (99.9%)
	Chain 3 had 999

In [33]:
res_singleGene

{'status': 'warn',
 'gene': 'BRAF',
 'N': 45,
 'n_aneup': 30,
 'cna': 'all',
 'mean_b_scaling': 0.62375673725,
 'sd_b_scaling': 0.08434806692365067,
 'mean_b_deviation': -0.25922642451999994,
 'sd_b_deviation': 0.09785032407187337,
 'z_comp': -2.649213755588505,
 'p_value': 0.008067928481445552,
 'mean_phi': 10.4982187,
 'Rhat_b_deviation': 1.33109,
 'ess_b_deviation': 8.08156,
 'covar_levels': ['ALL']}

#### Parallel run on multiple genes

In [10]:
# One-time split
gene_groups = {g: gdf for g, gdf in df_long.groupby("gene", sort=False)}

def run_gene(g):
    return fit_one_gene(
        gene_groups[g],
        model,
        cna="all",
        et=0.15,
        min_aneup=5,
        chains=4,
        iter_warmup=1000,
        iter_sampling=1000,
        adapt_delta=0.99,
        max_treedepth=15,
        show_progress=False,
        seed=1
    )

genes = list(gene_groups.keys())

results = Parallel(n_jobs=8, backend="loky")(
    delayed(run_gene)(g) for g in genes
)

res_df = pd.DataFrame(results)

#### Downstream results analysis

In [9]:
# Load results data
DATA_PATH = "/Users/katsiarynadavydzenka/Documents/PhD_AI/DeConveil/datasets/tcga_luad/"
res_df = pd.read_csv(os.path.join(DATA_PATH, "results_nb.csv"))
res_df.head()

,Unnamed: 0,status,gene,N,n_aneup,cna,mean_b_scaling,sd_b_scaling,mean_b_deviation,sd_b_deviation,z_comp,p_value,mean_phi,Rhat_b_deviation,ess_b_deviation,covar_levels,reason
0,0,ok,DPM1,45.0,27.0,all,0.608314,0.108923,-0.212012,0.153794,-1.378551,0.168033,7.107502,1.00128,1646.39,['ALL'],NaN
1,1,ok,SCYL3,45.0,33.0,all,0.690713,0.118381,-0.286183,0.163735,-1.747848,0.080490,7.574430,1.00016,1504.34,['ALL'],NaN
2,2,ok,C1orf112,45.0,33.0,all,0.429658,0.125695,-0.072222,0.169412,-0.426309,0.669883,4.010699,1.00318,1507.24,['ALL'],NaN
3,3,ok,FGR,45.0,23.0,all,0.116563,0.051953,-0.296982,0.103110,-2.880252,0.003974,3.392739,1.00338,1064.70,['ALL'],NaN
4,4,ok,CFH,45.0,32.0,all,0.415805,0.149208,-0.280086,0.172990,-1.619092,0.105428,1.803770,1.00058,1778.69,['ALL'],NaN


In [11]:
# Post-process into DSG/DCG calls
res_pp = postprocess_nb_results(res_df, 
                                alpha=0.05, 
                                comp_thr=0.5)
res_pp.head()

,Unnamed: 0,status,gene,N,n_aneup,cna,mean_b_scaling,sd_b_scaling,mean_b_deviation,sd_b_deviation,...,mean_phi,Rhat_b_deviation,ess_b_deviation,covar_levels,reason,label_nb,aneup_frac,comp_score,signed_comp,shrunk_comp
0,0,ok,DPM1,45.0,27.0,all,0.608314,0.108923,-0.212012,0.153794,...,7.107502,1.00128,1646.39,['ALL'],NaN,DSG,0.600000,-0.348524,-0.348524,-0.289961
1,1,ok,SCYL3,45.0,33.0,all,0.690713,0.118381,-0.286183,0.163735,...,7.574430,1.00016,1504.34,['ALL'],NaN,DSG,0.733333,-0.414330,-0.414330,-0.380981
2,2,ok,C1orf112,45.0,33.0,all,0.429658,0.125695,-0.072222,0.169412,...,4.010699,1.00318,1507.24,['ALL'],NaN,DSG,0.733333,-0.168092,-0.168092,-0.055490
3,3,ok,FGR,45.0,23.0,all,0.116563,0.051953,-0.296982,0.103110,...,3.392739,1.00338,1064.70,['ALL'],NaN,DCG,0.511111,-2.547811,-2.547811,-2.537687
4,4,ok,CFH,45.0,32.0,all,0.415805,0.149208,-0.280086,0.172990,...,1.803770,1.00058,1778.69,['ALL'],NaN,DCG,0.711111,-0.673600,-0.673600,-0.602584
